# JSON generator

This is to pull the applied rules out of the theory in some json format.

This is really hacky because I didn't plan properly and now have to reconstruct from the prolog/clingo.

In [1]:
import amati_bea as amati

In [2]:
import pandas as pd

from sklearn.metrics import f1_score, cohen_kappa_score

In [3]:
from operator import itemgetter
import pickle
import datetime
import re
import glob

## Get the data

We'll just be able to pull out the relevant data from the saved theory.

In [4]:
with open(
    "/Users/alistair.willis/repos/bea-2026/3-way/theories/run_20260303/theory_64393e93-5585-42f9-ad22-ffa71219573b_2026-03-04_02:35:57.762552.pickle", "rb"
) as fIn:
    theory = pickle.load(fIn)

In [5]:
theory.keys()

dict_keys(['theory', 'original_question_id', 'question_id', 'all_questions_df', 'all_responses_df', 'all_text_df', 'rules'])

In [37]:
def get_json_from_rule(rule):
    if rule["selected_rule"][0] == "rule1":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    }
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule2":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                    {"pred": "precedes", "order": ["@1", "@2"]},
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule3":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "ki_rule1":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "reference_answer",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@2",
                    },
                    {
                        "pred": "contains",
                        "negated":True,
                        "document": "question",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@3",
                    },
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    else:
        assert(False)

In [39]:
[get_json_from_rule(r) for r in theory['theory'][:-1]]

[{'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"umwandeln"',
     'id': '@1'}],
   'prediction': 'correct'}},
 {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"anschaffung-herstellung"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"umwandlung"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"energie"',
     'id': '@1'},
    {'pred': 'contains',
     'document': 'student_response',
     'lemma': '"wandeln"',
     'id': '@2'},
    {'pred': 'precedes', 'order': ['@1', '@2']}],
   'prediction': 'correct'}},
 {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"produzieren"',
     'id': '@1'}],
   'prediction': 'partially_correct'}},
 {'amati':

In [36]:
get_json_from_rule(theory['theory'][3])

{'amati': {'literals': [{'pred': 'contains',
    'document': 'student_response',
    'lemma': '"energie"',
    'id': '@1'},
   {'pred': 'contains',
    'document': 'student_response',
    'lemma': '"wandeln"',
    'id': '@2'}],
  'prediction': 'correct'}}

In [35]:
theory['theory'][3]

{'result': 'OPTIMUM FOUND',
 'selected_rule': ('rule2', 'selected_rule(rule2)'),
 'predicted_grade': ('correct', 'predicted_grade(correct)'),
 'parameters': [(1, '"energie"', 'parameter(1,"energie")'),
  (2, '"wandeln"', 'parameter(2,"wandeln")')],
 'positive_covered': ['r_2866', 'r_6505', 'r_7705'],
 'negative_covered': [],
 'evaluations': {'precision': 100,
  'compression': 2,
  'coverage': 3,
  'accuracy': 100},
 'positive_df':    response_id question_id              score  covered
 0       r_0193        q_18  partially_correct    False
 4       r_0410        q_18  partially_correct    False
 6       r_0456        q_18  partially_correct    False
 8       r_0585        q_18          incorrect    False
 9       r_0627        q_18  partially_correct    False
 10      r_0631        q_18  partially_correct    False
 11      r_0632        q_18  partially_correct    False
 15      r_0804        q_18  partially_correct    False
 16      r_0878        q_18  partially_correct    False
 20   

Let's find a trial response. First, need to pull out the question for that theory:

In [23]:
theory["question_id"]

'q_18'

And then get the associated responses:

In [24]:
theory["all_responses_df"].query("`question_id`==@theory['question_id']").query(
    "`partition`=='trial'"
).head()

,original_response_id,response_id,question_id,response,score,partition
2102,07cf8c34-4a35-4d99-8ad4-8e1358fbd089,r_0379,q_18,Die Bewegungsenergie des Windes wird zunächst ...,correct,trial
2103,faacc838-2de9-40d2-99d0-dec674a1ef50,r_1842,q_18,-Überflüssiger Strom wird effizient genutzt -n...,incorrect,trial
2104,f615be76-98ba-4c39-91f2-0b4c5354759a,r_2248,q_18,Die Bewegungsenergie fließt in eine erneuerbar...,correct,trial
2105,d700fe2d-d7e1-4792-abe0-b193289e8c6e,r_2567,q_18,die WIndenergie wird in Gas um gewandelt\n,partially_correct,trial
2106,a2857c1f-8410-4c22-9876-c64b58ccae0d,r_2818,q_18,Energie wird in einem Elektrolyt Kraftwerk zu ...,partially_correct,trial


So let's just bag the first of those:

In [25]:
TEST_RESPONSE_ID = (
    theory["all_responses_df"]
    .query("`question_id`==@theory['question_id']")
    .query("`partition`=='trial'")
    .iloc[0, 0]
)

TEST_RESPONSE_ID

'07cf8c34-4a35-4d99-8ad4-8e1358fbd089'

Reverse ferret, let's apply to the trial set:

In [26]:
def apply_trial_set(theory):

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id").query(
        "`partition`=='trial'"
    )

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["partially_correct_id"],
        question_ss["incorrect_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_df = amati.evaluate_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    )

    return {"theory": theory, "results": results_df, "compare": compare_df}

In [27]:
o=apply_trial_set(theory)

In [28]:
o['results']

,0,1,2,3,4,5,6,7
response_id,,,,,,,,
r_0379,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2248,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2818,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3719,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4642,correct,NaN,incorrect,NaN,NaN,NaN,NaN,NaN
r_4743,correct,NaN,NaN,correct,NaN,NaN,NaN,NaN
r_4750,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Apply to all theories

In [ ]:
outputs_ls=[]

for f in glob.glob("theories/run_20260303/*"):
    with open(f, 'rb') as fIn:
        theory=pickle.load(fIn)
    outputs_ls.append(apply_trial_set(theory))

/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_97070/322029500.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_97070/322029500.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_97070/322029500.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bf

And now we should be able to get an overall assessment...

In [42]:
outputs_ls[1]['results']

,0,1,2,3,4,5,6,7,8,9,10,11
response_id,,,,,,,,,,,,
r_0216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1288,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,correct,NaN,NaN,NaN
r_3146,NaN,correct,NaN,NaN,partially_correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3810,NaN,NaN,NaN,NaN,NaN,partially_correct,NaN,NaN,NaN,NaN,correct,NaN
r_3960,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4954,NaN,NaN,NaN,NaN,NaN,partially_correct,NaN,NaN,NaN,NaN,NaN,NaN
r_5360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct,NaN,NaN,NaN
r_6644,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,correct,NaN,correct,correct


In [86]:
outputs_ls[2]['results']

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
response_id,,,,,,,,,,,,,,,,,,,,,
r_0285,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0706,NaN,NaN,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1249,NaN,NaN,NaN,correct,NaN,NaN,NaN,NaN,NaN,correct,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1345,correct,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2003,NaN,correct,correct,NaN,NaN,correct,NaN,NaN,NaN,correct,...,NaN,NaN,partially_correct,correct,NaN,NaN,NaN,NaN,NaN,partially_correct
r_2144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3880,NaN,NaN,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4363,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,correct,NaN,NaN,NaN,NaN,partially_correct


First, get a big table of all the predictions:

In [87]:
predictions_df=pd.concat([o['results'] for o in outputs_ls])
predictions_df

,0,1,2,3,4,5,6,7,8,9,...,35,36,37,38,39,40,41,42,43,44
response_id,,,,,,,,,,,,,,,,,,,,,
r_2959,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_7542,NaN,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1288,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,correct,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
r_5297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_6723,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_7051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
# Let's define an ordered category for the prediction:

prediction_type = pd.CategoricalDtype(
    categories=['incorrect', 'partially_correct', 'correct'],
    ordered=True
)
prediction_type

CategoricalDtype(categories=['incorrect', 'partially_correct', 'correct'], ordered=True, categories_dtype=object)

And we can use this type on the predictions DataFrame:

In [89]:
predictions_df=predictions_df.astype(prediction_type)


Now, an interesting comparison might be whether we choose the first prediction or the last prediction. ie. Do later predictions override the earlier ones, or should the first prediction stand?

In [90]:
predict_first_ss=predictions_df.bfill(axis='columns').iloc[:, 0].fillna('incorrect')
predict_first_ss

response_id
r_2959            incorrect
r_7542            incorrect
r_0216            incorrect
r_0773            incorrect
r_1288              correct
                ...        
r_5297    partially_correct
r_6723              correct
r_7051            incorrect
r_7292              correct
r_7474              correct
Name: 0, Length: 827, dtype: category
Categories (3, object): ['incorrect' < 'partially_correct' < 'correct']

In [91]:
predict_last_ss=predictions_df.ffill(axis='columns').iloc[:, -1].fillna('incorrect')
predict_last_ss

response_id
r_2959            incorrect
r_7542            incorrect
r_0216            incorrect
r_0773            incorrect
r_1288              correct
                ...        
r_5297    partially_correct
r_6723              correct
r_7051            incorrect
r_7292              correct
r_7474              correct
Name: 44, Length: 827, dtype: category
Categories (3, object): ['incorrect' < 'partially_correct' < 'correct']

And now we need to do a comparison with the ground truth results.

In [92]:
results_df = (
    theory["all_responses_df"]
    .query("`partition`=='trial'")
    .set_index("response_id")[["score"]]
    .rename({"score": "actual"}, axis="columns")
    .astype(prediction_type)
    .assign(predict_first=predict_first_ss, predict_last=predict_last_ss)
)

results_df

,actual,predict_first,predict_last
response_id,,,
r_0424,incorrect,incorrect,incorrect
r_0651,incorrect,incorrect,incorrect
r_0654,incorrect,incorrect,incorrect
r_1176,incorrect,incorrect,incorrect
r_1719,partially_correct,incorrect,incorrect
...,...,...,...
r_3908,incorrect,incorrect,incorrect
r_5695,incorrect,incorrect,incorrect
r_5893,correct,incorrect,incorrect


In [93]:
results_df.query('`predict_first`!=`predict_last`')

,actual,predict_first,predict_last
response_id,,,
r_3238,partially_correct,incorrect,partially_correct
r_4047,incorrect,incorrect,partially_correct
r_5891,partially_correct,partially_correct,incorrect
r_6376,correct,correct,partially_correct
r_2299,correct,partially_correct,correct
r_6337,incorrect,partially_correct,correct
r_7129,correct,partially_correct,correct
r_7345,partially_correct,correct,partially_correct
r_4509,correct,partially_correct,correct


In [94]:
f1_score(results_df['actual'], results_df['predict_first'], average='weighted')

0.5567282751992583

In [95]:
f1_score(results_df['actual'], results_df['predict_last'], average='weighted')

0.5359536597537067

OK, predict first seems slightly better, as expected, although they're both pretty dire. Now let's do the evaluation with CWK:

In [ ]:
cohen_kappa_score(results_df['actual'], results_df['predict_first'], weights='quadratic')

0.3795158874519998

Ugghh... Although I wonder what the f1 score is:

In [98]:
f1_score(results_df['actual'], results_df['predict_first'], average='weighted')

0.5567282751992583

That's hardly going to set the world alight...